# Stage 2 — Mini local-agent batch (Colab, no Docker)

Runs the **mini bug-fix sandbox** on Colab CPU: real multi-step agent loops on
hand-written Python tasks, calling Qwen3-8B through an OpenAI-compatible API.
Output is SWE-agent-compatible `.traj` JSON → ingest → `normalized.zip` for projection.

**Runtime:** **CPU is fine** (Runtime → Change runtime type → CPU). No GPU needed here.

## Pipeline (3 Colab tabs total)

1. **[serve_qwen_colab.ipynb](https://colab.research.google.com/github/abdelmagid07/latent_failiure_prediction/blob/main/stage2/notebooks/serve_qwen_colab.ipynb)** — A100, run all cells, copy `MODEL_API_BASE`, keep tab open.
2. **This notebook** — CPU, mini batch + ingest, download `normalized.zip`.
3. **[project_and_analyze_colab.ipynb](https://colab.research.google.com/github/abdelmagid07/latent_failiure_prediction/blob/main/stage2/notebooks/project_and_analyze_colab.ipynb)** — A100, upload proxy axis + `normalized.zip`, project + de-risk analyses.

**Label for Jonas:** mini-agent noise feasibility probe — not full SWE-bench scale.

### Configuration

Paste the tunnel URL from **serve_qwen_colab.ipynb** (must include `/v1`).

Set `SMOKE_ONLY = True` for a 1-instance smoke test; set `False` for the full 12-instance batch.

In [ ]:
# Paste from serve_qwen_colab.ipynb cell 5 output:
MODEL_API_BASE = "https://YOUR-TUNNEL.trycloudflare.com/v1"
MODEL_API_KEY = "EMPTY"
MODEL_NAME = "Qwen3-8B"  # vLLM served name — NOT hosted_vllm/...

SMOKE_ONLY = True   # True = 1 instance (~5 min). False = full 12 (~30-60 min).
MAX_STEPS = 25      # agent steps per instance

assert MODEL_API_BASE.startswith("https://"), "Set MODEL_API_BASE from the serve notebook"
assert "/v1" in MODEL_API_BASE, "URL must include /v1 suffix"
print("API:", MODEL_API_BASE)
print("Mode:", "smoke (1 instance)" if SMOKE_ONLY else "full batch (12 instances)")

In [ ]:
# Clone + install (CPU-only deps — no torch/GPU needed).
import os
if not os.path.isdir('/content/latent_failiure_prediction'):
    !git clone https://github.com/abdelmagid07/latent_failiure_prediction.git
else:
    %cd /content/latent_failiure_prediction
    !git pull origin main

%cd /content/latent_failiure_prediction
!git log --oneline -1
!pip install -q -e stage1 -e "stage2[mini]"

In [ ]:
# Preflight: confirm the remote vLLM endpoint is reachable.
import urllib.request, urllib.error, json

url = MODEL_API_BASE.rstrip('/') + '/models'
req = urllib.request.Request(url, headers={"Authorization": f"Bearer {MODEL_API_KEY}"})
try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        models = json.loads(resp.read())
    print("Endpoint OK:", models)
except urllib.error.URLError as e:
    raise RuntimeError(
        f"Cannot reach {url}. Is serve_qwen_colab.ipynb still running?\n{e}"
    ) from e

In [ ]:
# Offline sanity: buggy mini repos must fail pytest before the agent runs.
%cd /content/latent_failiure_prediction/stage2
!pytest tests/test_mini_catalog.py -q

In [ ]:
# Run the mini agent batch.
import os
import subprocess
import sys
from datetime import datetime

os.environ["MODEL_API_BASE"] = MODEL_API_BASE
os.environ["MODEL_API_KEY"] = MODEL_API_KEY
os.environ["MODEL_NAME"] = MODEL_NAME

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = f"data/trajectories/mini_run_{ts}"

cmd = [
    sys.executable, "-m", "stage2.mini.run_batch",
    "--output-dir", OUTPUT_DIR,
    "--api-base", MODEL_API_BASE,
    "--api-key", MODEL_API_KEY,
    "--model", MODEL_NAME,
    "--max-steps", str(MAX_STEPS),
    "--skip-preflight",
]
if SMOKE_ONLY:
    cmd += ["--instance-id", "mini_eventbus_001"]
else:
    cmd += ["--instances", "config/mini_instances.txt"]

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True, cwd="/content/latent_failiure_prediction/stage2")
print("\nRun directory:", OUTPUT_DIR)

In [ ]:
# Inspect results (resolved vs failed).
import json, glob
from pathlib import Path

if 'OUTPUT_DIR' not in dir():
    runs = sorted(glob.glob('data/trajectories/mini_run_*'))
    assert runs, 'No mini run found — run the batch cell first'
    OUTPUT_DIR = runs[-1]

results_path = Path(OUTPUT_DIR) / 'results.json'
assert results_path.exists(), f'Missing {results_path}'
results = json.loads(results_path.read_text())
print(f"Resolved:   {len(results.get('resolved_ids', []))}  {results.get('resolved_ids', [])}")
print(f"Unresolved: {len(results.get('unresolved_ids', []))}  {results.get('unresolved_ids', [])}")
print(f"Traj files: {len(list(Path(OUTPUT_DIR).glob('*.traj')))}")

### Ingest → normalized JSON

Only run this after a successful batch (smoke or full). For the **full de-risk**,
re-run the batch cell with `SMOKE_ONLY = False`, then run ingest + download below.

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable, "-m", "stage2.trajectories.ingest_batch",
        "--traj-dir", OUTPUT_DIR,
        "--output-dir", "data/normalized",
    ],
    check=True,
    cwd="/content/latent_failiure_prediction/stage2",
)

In [ ]:
# Label check before uploading to the projection notebook.
import json, glob
outcomes = [json.load(open(p)).get('outcome') for p in glob.glob('data/normalized/*.json')]
n_succ = sum(1 for o in outcomes if o == 1)
n_fail = sum(1 for o in outcomes if o == 0)
print(f'success={n_succ}  failure={n_fail}  total={len(outcomes)}')
if n_succ == 0 or n_fail == 0:
    print('WARNING: only one outcome class — de-risk plots will be weak. Consider re-running batch.')

In [ ]:
# Zip normalized trajectories for project_and_analyze_colab.ipynb.
import shutil
from google.colab import files

zip_path = '/content/normalized.zip'
shutil.make_archive('/content/normalized', 'zip', 'data/normalized')
print('Created', zip_path)
files.download(zip_path)

In [ ]:
# Optional: download raw results + manifest for your records.
from google.colab import files
from pathlib import Path

for name in ['results.json', 'run_manifest.json']:
    p = Path(OUTPUT_DIR) / name
    if p.exists():
        files.download(str(p))

### Next step

Open **[project_and_analyze_colab.ipynb](https://colab.research.google.com/github/abdelmagid07/latent_failiure_prediction/blob/main/stage2/notebooks/project_and_analyze_colab.ipynb)** on an **A100** runtime.

Upload:
- `value_axis_proxy.npy` + `axis_manifest_proxy.json` (from Stage 1)
- `normalized.zip` (downloaded above)

Run all cells → download the four Jonas deliverables.

In [ ]:
# Optional: save artifacts to Google Drive so a disconnect doesn't lose them.
# from google.colab import drive
# drive.mount('/content/drive')
# import shutil
# dest = '/content/drive/MyDrive/stage2_mini_run'
# shutil.copytree(OUTPUT_DIR, dest + '/trajectories', dirs_exist_ok=True)
# shutil.copytree('data/normalized', dest + '/normalized', dirs_exist_ok=True)
# shutil.copy('/content/normalized.zip', dest + '/normalized.zip')
# print('Saved to', dest)